In [2]:
from pyspark.sql.functions import *

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

gold_sales = (spark.table("silver.order_items").alias("oi")

    .join(spark.table("silver.orders").alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "left")

    .join(spark.table("silver.customers").alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left")

    .join(spark.table("silver.products").alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left")

    .join(spark.table("silver.sellers").alias("s"),
        col("oi.seller_id") == col("s.seller_id"),
        "left")

    .join(spark.table("silver.payments").alias("pay"),
        col("oi.order_id") == col("pay.order_id"),
        "left")

    .select(
        col("oi.order_id"),
        col("oi.order_item_id"),

        col("o.order_status"),
        col("o.order_purchase_timestamp"),
        col("o.order_year"),
        col("o.order_month"),

        col("c.customer_id"),
        col("c.customer_city"),
        col("c.customer_state"),

        col("p.product_id"),
        col("p.product_category_name_english"),

        col("s.seller_id"),
        col("s.seller_city"),
        col("s.seller_state"),

        col("pay.payment_type"),
        col("pay.payment_installments"),

        col("oi.price"),
        col("oi.freight_value"),
        (col("oi.price") + col("oi.freight_value")).alias("total_sale_amount"),

        current_timestamp().alias("gold_load_timestamp")))

gold_sales.write.mode("overwrite").format("delta").saveAsTable("gold.sales_fact")

print(f"Gold table created: {gold_sales.count()} rows")


StatementMeta(, 1a0e6755-b3d8-463b-a45d-5189170a3623, 4, Finished, Available, Finished, False)

Gold table created: 117604 rows


In [3]:
spark.sql("""OPTIMIZE gold.sales_fact""")

spark.sql("""VACUUM gold.sales_fact RETAIN 168 HOURS""")

print("Optimization completed.")

StatementMeta(, 1a0e6755-b3d8-463b-a45d-5189170a3623, 5, Finished, Available, Finished, False)

Optimization completed.


In [4]:
gold_sales.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .format("delta") \
    .saveAsTable("gold.sales_fact")

StatementMeta(, 1a0e6755-b3d8-463b-a45d-5189170a3623, 6, Finished, Available, Finished, False)

In [5]:
spark.sql("""DESCRIBE DETAIL gold.sales_fact""").select("format", "partitionColumns", "numFiles").show(truncate=False)

StatementMeta(, 1a0e6755-b3d8-463b-a45d-5189170a3623, 7, Finished, Available, Finished, False)

+------+-------------------------+--------+
|format|partitionColumns         |numFiles|
+------+-------------------------+--------+
|delta |[order_year, order_month]|24      |
+------+-------------------------+--------+

